# 07 Limitations And Data Budget

This notebook answers the final practical question: how much headroom is likely left, and where does the current thesis package still remain scientifically limited?

**Questions answered here**
- Are we still on the steep part of the paired-label curve?
- How much performance is lost under family-unseen transfer?
- Which limitations are methodological, and which are data-limited?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"
if not SRC.exists():
    raise FileNotFoundError(f"Expected thesis src at {SRC}; run notebooks from thesis/notebooks")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()

fig_dir, table_dir = notebook_output_dirs("07_limitations_and_data_budget")
bundle = get_limitations_bundle()
fpos_progress = build_benchmark_progress_table("fpos")
fmiss_progress = build_benchmark_progress_table("fmiss")


## 0. Pinning the best results for headroom analysis

The limitation analysis is only meaningful relative to a specific set of results. We pin both best models and all baselines here so the headroom section uses exactly the same recipe results as the main progression notebooks.

In [ ]:
PROTOCOL = "recording_disjoint_main"

# Baselines
r_fpos_dummy  = load_or_run_experiment("fpos_dummy",             "fpos",  PROTOCOL)
r_fmiss_dummy = load_or_run_experiment("fmiss_dummy",            "fmiss", PROTOCOL)
r_fpos_basic  = load_or_run_experiment("fpos_hybrid_plus_paired","fpos",  PROTOCOL)
r_fmiss_basic = load_or_run_experiment("fmiss_hybrid_residual",  "fmiss", PROTOCOL)
# Best models
r_fpos_best   = load_or_run_experiment("fpos_waveform_winner",   "fpos",  PROTOCOL)
r_fmiss_best  = load_or_run_experiment("fmiss_reduced_latent",   "fmiss", PROTOCOL)

def _m(result):
    row = result.metrics.iloc[0] if not result.metrics.empty else {}
    return {"r2": float(row.get("r2", float("nan"))), "mae": float(row.get("mae", float("nan")))}

ceiling_summary = pd.DataFrame([
    {"target": "fpos",  "dummy_r2": _m(r_fpos_dummy)["r2"],
     "basic_transfer_r2": _m(r_fpos_basic)["r2"],
     "best_r2": _m(r_fpos_best)["r2"],
     "gain_over_basic": _m(r_fpos_best)["r2"] - _m(r_fpos_basic)["r2"]},
    {"target": "fmiss", "dummy_r2": _m(r_fmiss_dummy)["r2"],
     "basic_transfer_r2": _m(r_fmiss_basic)["r2"],
     "best_r2": _m(r_fmiss_best)["r2"],
     "gain_over_basic": _m(r_fmiss_best)["r2"] - _m(r_fmiss_basic)["r2"]},
])
display(ceiling_summary)
save_table(ceiling_summary, table_dir, "thesis_best_results_ceiling_summary")

## 1. Label-budget evidence

The thesis should make the small-data regime visible. If the curve is still rising steeply, then the current benchmark is probably data-limited rather than fully saturated.


In [ ]:
display(bundle.label_budget)
save_table(bundle.label_budget, table_dir, "label_budget_summary")
fig, _ = plot_label_budget(bundle.label_budget, target="fpos")
save_figure(fig, fig_dir, "label_budget_fpos")
display(fig)
plt.close(fig)


In [ ]:
fig, _ = plot_label_budget(bundle.label_budget, target="fmiss")
save_figure(fig, fig_dir, "label_budget_fmiss")
display(fig)
plt.close(fig)


## 2. Family-held-out robustness

The main benchmark optimizes recording-disjoint transfer. This section reports the explicit penalty for requiring generalization to an unseen paired family and places both protocols side-by-side for direct interpretation.


In [ ]:
display(bundle.lofo_aggregate)
display(bundle.lofo_family)
save_table(bundle.lofo_aggregate, table_dir, "leave_one_family_aggregate")
save_table(bundle.lofo_family, table_dir, "leave_one_family_by_family")
fig, _, _ = plot_group_metric(bundle.lofo_family, group_col="held_out_family", metric="r2", title="Leave-one-family-out `fpos` R²")
save_figure(fig, fig_dir, "leave_one_family_r2")
display(fig)
plt.close(fig)

# Explicit protocol comparison for the thesis best `fpos` model.
winner_variant = "wf_embed_anchor_stack_xgboost"
winner_recording = _m(r_fpos_best)
winner_lofo = bundle.lofo_aggregate.loc[bundle.lofo_aggregate["variant_id"].astype(str) == winner_variant].copy()
if winner_lofo.empty:
    raise RuntimeError(f"Missing family-disjoint aggregate row for {winner_variant}")
winner_lofo_row = winner_lofo.iloc[0]

protocol_compare = pd.DataFrame([
    {
        "target": "fpos",
        "model": "Waveform winner",
        "protocol": "recording_disjoint_main",
        "r2": float(winner_recording["r2"]),
        "mae": float(winner_recording["mae"]),
    },
    {
        "target": "fpos",
        "model": "Waveform winner",
        "protocol": "family_disjoint_lofo",
        "r2": float(winner_lofo_row["macro_r2"]),
        "mae": float(winner_lofo_row["macro_mae"]),
    },
])
recording_r2 = float(protocol_compare.loc[protocol_compare["protocol"] == "recording_disjoint_main", "r2"].iloc[0])
recording_mae = float(protocol_compare.loc[protocol_compare["protocol"] == "recording_disjoint_main", "mae"].iloc[0])
protocol_compare["delta_r2_vs_recording"] = protocol_compare["r2"] - recording_r2
protocol_compare["delta_mae_vs_recording"] = protocol_compare["mae"] - recording_mae
display(protocol_compare)
save_table(protocol_compare, table_dir, "fpos_protocol_comparison_winner")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(protocol_compare["protocol"], protocol_compare["r2"], color=["#2957a4", "#d98e04"])
axes[0].set_title("Waveform winner R² by protocol")
axes[0].set_ylabel("R²")
axes[0].tick_params(axis="x", rotation=18)
axes[1].bar(protocol_compare["protocol"], protocol_compare["mae"], color=["#2957a4", "#d98e04"])
axes[1].set_title("Waveform winner MAE by protocol")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis="x", rotation=18)
fig.tight_layout()
save_figure(fig, fig_dir, "fpos_protocol_comparison_winner")
display(fig)
plt.close(fig)

# Family-level side-by-side protocol view for the same model.
winner_family_recording = build_family_summary("fpos_waveform_winner", "fpos", PROTOCOL)
if winner_family_recording is not None and not winner_family_recording.empty:
    rec_family = winner_family_recording[["study_set", "row_count", "r2", "mae"]].rename(
        columns={
            "row_count": "recording_rows",
            "r2": "recording_disjoint_r2",
            "mae": "recording_disjoint_mae",
        }
    )
    lofo_family = bundle.lofo_family.loc[bundle.lofo_family["variant_id"].astype(str) == winner_variant, ["held_out_family", "n_test_rows", "r2", "mae"]].rename(
        columns={
            "held_out_family": "study_set",
            "n_test_rows": "family_disjoint_rows",
            "r2": "family_disjoint_r2",
            "mae": "family_disjoint_mae",
        }
    )
    family_protocol_compare = rec_family.merge(lofo_family, on="study_set", how="inner")
    family_protocol_compare["r2_gap_family_minus_recording"] = (
        family_protocol_compare["family_disjoint_r2"] - family_protocol_compare["recording_disjoint_r2"]
    )
    family_protocol_compare["mae_gap_family_minus_recording"] = (
        family_protocol_compare["family_disjoint_mae"] - family_protocol_compare["recording_disjoint_mae"]
    )
    display(family_protocol_compare.sort_values("r2_gap_family_minus_recording"))
    save_table(family_protocol_compare, table_dir, "fpos_family_protocol_comparison_by_family")

    plot_df = family_protocol_compare.sort_values("study_set").reset_index(drop=True)
    y = np.arange(len(plot_df))
    h = 0.38
    fig, ax = plt.subplots(figsize=(10, max(4, 0.7 * len(plot_df))))
    ax.barh(y - h / 2, plot_df["recording_disjoint_r2"], height=h, color="#2957a4", label="Recording-disjoint")
    ax.barh(y + h / 2, plot_df["family_disjoint_r2"], height=h, color="#d98e04", label="Family-disjoint")
    ax.set_yticks(y, plot_df["study_set"])
    ax.set_xlabel("R²")
    ax.set_title("Waveform winner: family-level R² by protocol")
    ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, fig_dir, "fpos_family_protocol_comparison_r2")
    display(fig)
    plt.close(fig)

# Coverage note so protocol availability is explicit by target.
coverage = pd.DataFrame([
    {
        "target": "fpos",
        "recording_disjoint_available": True,
        "family_disjoint_available": bool((bundle.lofo_family["variant_id"].astype(str).str.contains("wf_embed_anchor_stack_xgboost")).any()),
        "family_disjoint_source": "fpos_leave_one_family_{aggregate,family}.csv",
    },
    {
        "target": "fmiss",
        "recording_disjoint_available": True,
        "family_disjoint_available": bool((bundle.lofo_family["variant_id"].astype(str).str.contains("fmiss")).any()),
        "family_disjoint_source": "not present in frozen_inputs/benchmarks",
    },
])
display(coverage)
save_table(coverage, table_dir, "protocol_coverage_by_target")


## 3. What the current results imply about headroom

The table below puts the current best results next to their simpler transfer baselines. It is not a theoretical upper bound, but it does show that there is still room between “basic transfer” and “current best”.


In [ ]:
basic_fpos = fpos_progress.loc[fpos_progress["recipe_id"] == "fpos_hybrid_plus_paired"].iloc[0]
final_fpos = fpos_progress.sort_values("r2", ascending=False).iloc[0]
basic_fmiss = fmiss_progress.loc[fmiss_progress["recipe_id"] == "fmiss_hybrid_residual"].iloc[0]
final_fmiss = fmiss_progress.sort_values("r2", ascending=False).iloc[0]
headroom = pd.DataFrame([
    {"target": "fpos", "basic_transfer_r2": basic_fpos["r2"], "current_best_r2": final_fpos["r2"], "gain_r2": final_fpos["r2"] - basic_fpos["r2"]},
    {"target": "fmiss", "basic_transfer_r2": basic_fmiss["r2"], "current_best_r2": final_fmiss["r2"], "gain_r2": final_fmiss["r2"] - basic_fmiss["r2"]},
])
display(headroom)
save_table(headroom, table_dir, "headroom_summary")


In [ ]:
display(Markdown(
    """
## Limitation summary

- The paired labeled pool is still small enough that label-budget curves matter.
- Side-by-side protocol plots show the family-disjoint penalty relative to recording-disjoint performance for the same `fpos` winner.
- Leave-one-family-out performance remains clearly worse than the main benchmark, so family-unseen robustness is not solved.
- `fpos` has both protocol reports in frozen benchmark artifacts; `fmiss` currently has recording-disjoint only in this package.
- The most plausible next gains should come from richer spatial/template/drift information, not from more generic backend tinkering.
"""
))
